In [6]:
# ============================================================
# ML MODEL 2: APPOINTMENT NO-SHOW PREDICTION
# ============================================================

# Import required libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)


# ------------------------------------------------------------
# 1. LOAD CLEANED DATA
# ------------------------------------------------------------

# Load the cleaned appointment dataset.
# The data was already validated and cleaned during EDA.

file_path = r"D:\Hospital-Operations-Intelligence\data\processed\appointments\appointments_clean.csv"

appointments = pd.read_csv(
    file_path,
    parse_dates=[
        "appointment_date",
        "booking_date"
    ]
)

print("Dataset shape:", appointments.shape)


# ------------------------------------------------------------
# 2. CREATE TARGET
# ------------------------------------------------------------

# 1 = No-Show
# 0 = Other appointment outcome

appointments["no_show"] = (
    appointments["appointment_status"] == "No-Show"
).astype(int)

print("\nNo-Show Distribution:")
print(appointments["no_show"].value_counts())


# ------------------------------------------------------------
# 3. CREATE FEATURES
# ------------------------------------------------------------

# Calculate the number of days between booking and appointment.

appointments["booking_lead_days"] = (
    appointments["appointment_date"]
    - appointments["booking_date"]
).dt.days

# Extract day of week.
# Monday = 0, Sunday = 6.

appointments["appointment_dayofweek"] = (
    appointments["appointment_date"].dt.dayofweek
)

# Extract month.

appointments["appointment_month"] = (
    appointments["appointment_date"].dt.month
)

# Group booking lead time into meaningful categories.

appointments["booking_lead_group"] = pd.cut(
    appointments["booking_lead_days"],
    bins=[-1, 2, 7, 14, float("inf")],
    labels=[
        "Very Short",
        "Short",
        "Medium",
        "Long"
    ]
)


# ------------------------------------------------------------
# 4. SELECT FEATURES
# ------------------------------------------------------------

# appointment_status is excluded because it contains
# the actual outcome we are trying to predict.
#
# appointment_id and patient_id are excluded because they
# are identifiers and are not useful predictive features
# for this first model.

features = [
    "department_id",
    "appointment_type",
    "booking_lead_group",
    "wait_time_days",
    "booking_lead_days",
    "appointment_dayofweek",
    "appointment_month"
]

X = appointments[features]
y = appointments["no_show"]


# ------------------------------------------------------------
# 5. DEFINE FEATURE TYPES
# ------------------------------------------------------------

categorical_features = [
    "department_id",
    "appointment_type",
    "booking_lead_group"
]

numerical_features = [
    "wait_time_days",
    "booking_lead_days",
    "appointment_dayofweek",
    "appointment_month"
]


# ------------------------------------------------------------
# 6. PREPROCESSING
# ------------------------------------------------------------

# Fill missing numerical values with the median.
# Fill missing categorical values with the most frequent value.
# Convert categorical features into numerical values.

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numerical",
            SimpleImputer(strategy="median"),
            numerical_features
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "encoder",
                    OneHotEncoder(handle_unknown="ignore")
                )
            ]),
            categorical_features
        )
    ]
)


# ------------------------------------------------------------
# 7. TRAIN / TEST SPLIT
# ------------------------------------------------------------

# 80% of the data is used for training.
# 20% is used for testing.

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data:", X_test.shape)


# ------------------------------------------------------------
# 8. CREATE RANDOM FOREST MODEL
# ------------------------------------------------------------

# Random Forest is used because it can learn
# non-linear relationships between appointment features
# and No-Show risk.

no_show_model = Pipeline([
    (
        "preprocessing",
        preprocessor
    ),
    (
        "classifier",
        RandomForestClassifier(
            n_estimators=100,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        )
    )
])


# ------------------------------------------------------------
# 9. TRAIN MODEL
# ------------------------------------------------------------

no_show_model.fit(
    X_train,
    y_train
)

print("\nNo-Show model training completed successfully!")


# ------------------------------------------------------------
# 10. MAKE PREDICTIONS
# ------------------------------------------------------------

y_pred = no_show_model.predict(X_test)

print("Predictions generated successfully!")


# ------------------------------------------------------------
# 11. EVALUATE MODEL
# ------------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

precision = precision_score(
    y_test,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_test,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_test,
    y_pred,
    zero_division=0
)

print("\n========== NO-SHOW MODEL PERFORMANCE ==========")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))


# ------------------------------------------------------------
# 12. CONFUSION MATRIX
# ------------------------------------------------------------

print("\n========== CONFUSION MATRIX ==========")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


# ------------------------------------------------------------
# 13. CLASSIFICATION REPORT
# ------------------------------------------------------------

print("\n========== CLASSIFICATION REPORT ==========")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "Attended/Other",
            "No-Show"
        ],
        zero_division=0
    )
)

Dataset shape: (60000, 8)

No-Show Distribution:
no_show
0    55692
1     4308
Name: count, dtype: int64

Training data: (48000, 7)
Testing data: (12000, 7)

No-Show model training completed successfully!
Predictions generated successfully!

========== NO-SHOW MODEL PERFORMANCE ==========
Accuracy : 0.8789
Precision: 0.0773
Recall   : 0.0626
F1 Score : 0.0692

========== CONFUSION MATRIX ==========
[[10493   645]
 [  808    54]]

========== CLASSIFICATION REPORT ==========
                precision    recall  f1-score   support

Attended/Other       0.93      0.94      0.94     11138
       No-Show       0.08      0.06      0.07       862

      accuracy                           0.88     12000
     macro avg       0.50      0.50      0.50     12000
  weighted avg       0.87      0.88      0.87     12000

